# Boardroom LLM Council (Personality-Based)

Prototype a boardroom-style council where the same or different LLMs take on distinct personas and debate before a final synthesis.

This notebook uses an OpenAI-compatible client so you can plug in providers that expose OpenAI-style APIs.


## Provider Notes (Free-Tier Friendly)

- Groq offers OpenAI-compatible endpoints and hosts top open-weight models like `llama-3.3-70b-versatile`.
- Mistral offers a free tier for their API platform.
- Hugging Face provides small free monthly credits for inference providers.

Pick a provider you have free access to, set its API key and base URL below, and choose a model.


In [ ]:
# If needed:
# !pip install -q openai

import os
from typing import List, Dict

try:
    from openai import OpenAI
except Exception as e:
    raise ImportError("Please install the openai package: pip install openai") from e

# --- Configure your provider ---
# Example for Groq (OpenAI-compatible):
#   BASE_URL = "https://api.groq.com/openai/v1"
#   MODEL = "llama-3.3-70b-versatile"
#   API_KEY = os.environ.get("GROQ_API_KEY")
#
# Example for Mistral (if using an OpenAI-compatible proxy or gateway):
#   BASE_URL = "https://api.mistral.ai/v1"  (confirm OpenAI-compat in your setup)
#
# For any OpenAI-compatible provider, set BASE_URL and API_KEY accordingly.
BASE_URL = os.environ.get("LLM_BASE_URL", "https://api.groq.com/openai/v1")
MODEL = os.environ.get("LLM_MODEL", "llama-3.3-70b-versatile")
API_KEY = os.environ.get("LLM_API_KEY")

if not API_KEY:
    raise ValueError("Set LLM_API_KEY in your environment (or update the code to load your provider key).")

client = OpenAI(base_url=BASE_URL, api_key=API_KEY)

def call_llm(messages: List[Dict], model: str = MODEL, temperature: float = 0.4) -> str:
    resp = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
    )
    return resp.choices[0].message.content


## Define Personalities
Each persona has a role, background, incentives, and a speaking style.


In [ ]:
PERSONAS = [
    {
        "name": "Regional Sales Head - Karnataka",
        "background": "Owns quarterly revenue in Karnataka; incentivized to grow local market share.",
        "style": "Pragmatic, numbers-driven, optimistic about local expansion.",
        "focus": "Local customer demand, sales trends, speed of market penetration."
    },
    {
        "name": "Country-wide Distribution Head",
        "background": "Responsible for nationwide logistics efficiency and cost control.",
        "style": "Risk-aware, process-oriented, skeptical of fragmented networks.",
        "focus": "Unit economics, logistics complexity, vendor SLAs, scaling."
    },
    {
        "name": "CEO",
        "background": "Balances growth with capital efficiency and long-term strategy.",
        "style": "Strategic, asks for trade-offs and long-term ROI.",
        "focus": "Strategic positioning, capital allocation, risk profile."
    }
]


## Council Orchestration
We do two rounds: initial viewpoints, then rebuttals, then a CEO synthesis.


In [ ]:
def persona_system_prompt(p):
    return (
        f"You are {p['name']}.\n"
        f"Background: {p['background']}\n"
        f"Focus: {p['focus']}\n"
        f"Speaking style: {p['style']}\n"
        "Be concise and practical. Provide 3-5 bullet points.\n"
        "If you make assumptions, label them.\n"
    )

def ask_persona(p, question, prior_messages=None, temperature=0.4):
    messages = [{"role": "system", "content": persona_system_prompt(p)}]
    if prior_messages:
        messages.extend(prior_messages)
    messages.append({"role": "user", "content": question})
    return call_llm(messages, temperature=temperature)

def run_council(question, personas=PERSONAS):
    # Round 1: initial views
    round1 = {}
    for p in personas:
        round1[p['name']] = ask_persona(p, question)

    # Round 2: rebuttals using others' points
    round2 = {}
    for p in personas:
        other_points = "\n\n".join([f"{k}:\n{v}" for k, v in round1.items() if k != p['name']])
        rebuttal_q = (
            f"Question: {question}\n"
            f"Other viewpoints:\n{other_points}\n\n"
            "Respond with key rebuttals or alignments (3-5 bullets)."
        )
        round2[p['name']] = ask_persona(p, rebuttal_q, temperature=0.5)

    # Final synthesis by CEO
    ceo = [p for p in personas if 'CEO' in p['name']]
    ceo = ceo[0] if ceo else personas[-1]

    synthesis_input = "\n\n".join([
        f"{k} (Round 1):\n{v}" for k, v in round1.items()
    ] + [
        f"{k} (Round 2):\n{v}" for k, v in round2.items()
    ])

    synth_prompt = (
        f"You are {ceo['name']}.\n"
        "Synthesize a boardroom decision.\n"
        "Output format:\n"
        "Decision: <one sentence>\n"
        "Rationale: 3-5 bullets\n"
        "Risks: 2-3 bullets\n"
        "Next Steps: 3 bullets\n"
        "Use the discussion below:\n"
        f"{synthesis_input}"
    )

    final = ask_persona(ceo, synth_prompt, temperature=0.3)

    return round1, round2, final


## Example Run


In [ ]:
question = "Should Acme Corporation set up its own distribution network in Karnataka or outsource it?"

round1, round2, final = run_council(question)

print("=== Round 1 ===")
for k, v in round1.items():
    print(f"\n[{k}]\n{v}")

print("\n=== Round 2 ===")
for k, v in round2.items():
    print(f"\n[{k}]\n{v}")

print("\n=== Final Synthesis ===\n")
print(final)
